In [1]:
# Cell 1: FIRST RUN ONLY - Clone & Install Everything
# After running this cell, RESTART KERNEL, then SKIP to Cell 2

import os
import shutil

%cd /workspace

# Clean slate
if os.path.exists("InternVL"):
    print("Removing old InternVL directory...")
    shutil.rmtree("InternVL")

# Clone and setup
print("Cloning Blind-Assist/InternVL...")
!git clone https://github.com/Blind-Assist/InternVL.git
%cd InternVL
!git checkout walkvlm
%cd internvl_chat

# Install ALL dependencies in one go
print("\n📦 Installing dependencies...")
!pip install -q \
    "transformers==4.37.2" \
    "peft==0.10.0" \
    "accelerate<1" \
    "deepspeed>=0.13.5" \
    "timm==0.9.12" \
    "einops==0.6.1" \
    "sentencepiece==0.1.99" \
    "tokenizers==0.15.1" \
    "datasets" \
    "huggingface_hub" \
    "decord" \
    "wandb" \
    "opencv-python-headless" \
    "numpy==1.26.4" \
    "scipy" \
    "scikit-learn>=1.2.2" \
    "orjson" \
    "pyyaml" \
    "termcolor" \
    "yacs"

# Install bitsandbytes (CUDA 12.x compatible version)
!pip uninstall bitsandbytes -y 2>/dev/null || true
!pip install -q bitsandbytes>=0.43.0

# Install flash-attention
!pip install -q flash-attn --no-build-isolation

# Fix typing_extensions
!pip install -q --upgrade "typing_extensions>=4.12.0" pydantic pydantic-core

print("\n" + "="*50)
print("✅ Installation complete!")
print("="*50)
print("⚠️  NOW: Restart Kernel (Kernel → Restart)")
print("⚠️  THEN: Skip this cell, run Cell 2")
print("="*50)

/workspace
Cloning Blind-Assist/InternVL...
Cloning into 'InternVL'...


/usr/local/lib/python3.11/dist-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


remote: Enumerating objects: 3473, done.
remote: Counting objects: 100% (130/130), done.
remote: Compressing objects: 100% (73/73), done.
remote: Total 3473 (delta 83), reused 69 (delta 57), pack-reused 3343 (from 2)
Receiving objects: 100% (3473/3473), 39.57 MiB | 14.32 MiB/s, done.
Resolving deltas: 100% (2098/2098), done.
Updating files: 100% (904/904), done.
/workspace/InternVL
Branch 'walkvlm' set up to track remote branch 'walkvlm' from 'origin'.
Switched to a new branch 'walkvlm'
/workspace/InternVL/internvl_chat

📦 Installing dependencies...

[notice] A new release of pip is available: 24.2 -> 26.0
[notice] To update, run: python -m pip install --upgrade pip

[notice] A new release of pip is available: 24.2 -> 26.0
[notice] To update, run: python -m pip install --upgrade pip

[notice] A new release of pip is available: 24.2 -> 26.0
[notice] To update, run: python -m pip install --upgrade pip

[notice] A new release of pip is available: 24.2 -> 26.0
[notice] To update, run: pyth

In [1]:
# Cell 2: POST-RESTART - Setup Environment
# Run this AFTER kernel restart

import os
%cd /workspace/InternVL/internvl_chat

# Verify environment
print("📁 Current directory:", os.getcwd())
print("\n🔍 Checking installations...")

import torch
print(f"✅ PyTorch: {torch.__version__}")
print(f"✅ CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✅ CUDA version: {torch.version.cuda}")
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")

import bitsandbytes
print(f"✅ bitsandbytes: {bitsandbytes.__version__}")

import transformers
print(f"✅ transformers: {transformers.__version__}")

print("\n✅ Environment ready! Continue to next cell.")

/usr/local/lib/python3.11/dist-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


/workspace/InternVL/internvl_chat
📁 Current directory: /workspace/InternVL/internvl_chat

🔍 Checking installations...
✅ PyTorch: 2.4.1+cu124
✅ CUDA available: True
✅ CUDA version: 12.4
✅ GPU: NVIDIA A40
✅ bitsandbytes: 0.49.1
✅ transformers: 4.37.2

✅ Environment ready! Continue to next cell.


In [2]:
# Cell 3: Login to Services
import wandb
from huggingface_hub import login

print("🔑 Login to WandB:")
wandb.login()

print("\n🔑 Login to Hugging Face:")
login()

🔑 Login to WandB:


wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

  2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

  ········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: rpgkr3 (vlm-research) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin



🔑 Login to Hugging Face:


In [3]:
# Cell 4: Patch dist_utils.py
patch_code = """import os
import torch
import torch.distributed as dist

def init_dist(launcher='pytorch', backend='nccl', **kwargs):
    if 'SLURM_PROCID' in os.environ:
        _init_dist_slurm(backend, **kwargs)
    else:
        _init_dist_pytorch(backend, **kwargs)

def _init_dist_pytorch(backend, **kwargs):
    rank = int(os.environ.get('RANK', 0))
    local_rank = int(os.environ.get('LOCAL_RANK', 0))
    world_size = int(os.environ.get('WORLD_SIZE', 1))
    
    torch.cuda.set_device(local_rank)
    
    if not dist.is_initialized():
        dist.init_process_group(
            backend=backend,
            init_method='env://',
            world_size=world_size,
            rank=rank
        )
    return local_rank

def _init_dist_slurm(backend, port=None, **kwargs):
    proc_id = int(os.environ['SLURM_PROCID'])
    ntasks = int(os.environ['SLURM_NTASKS'])
    node_list = os.environ['SLURM_NODELIST']
    num_gpus = torch.cuda.device_count()
    torch.cuda.set_device(proc_id % num_gpus)
    
    import subprocess
    addr = subprocess.getoutput(f'scontrol show hostname {node_list} | head -n1')
    
    if port is not None:
        os.environ['MASTER_PORT'] = str(port)
    elif 'MASTER_PORT' not in os.environ:
        os.environ['MASTER_PORT'] = '29500'
    
    os.environ['MASTER_ADDR'] = addr
    os.environ['WORLD_SIZE'] = str(ntasks)
    os.environ['RANK'] = str(proc_id)
    
    dist.init_process_group(backend=backend)
    return proc_id % num_gpus
"""

with open('internvl/dist_utils.py', 'w') as f:
    f.write(patch_code)

print("✅ Patched internvl/dist_utils.py")

✅ Patched internvl/dist_utils.py


In [ ]:
# Cell 5: Prepare Data
DATASET_LIMIT = 8500  # Change this as needed

!python prepare_walk_data.py --limit {DATASET_LIMIT}

# Verify data
print("\n📊 Data verification:")
!echo "Train samples:" && wc -l < data/walk_vlm/walk_train.jsonl
!echo "Val samples:" && wc -l < data/walk_vlm/walk_val.jsonl
!echo "\nSample images:" && ls data/walk_vlm/images | head -5

Loading dataset: blind-assist/walk-train (Streaming Mode)...
Resolving data files: 100%|██████████████| 8582/8582 [00:00<00:00, 40941.12it/s]
⚠️ LIMITING dataset to first 8500 videos.
📥 Collecting video metadata...
Scanning videos: 100%|█████████████████████| 8500/8500 [00:12<00:00, 657.59it/s]
✅ Found 8500 valid videos
📊 Video split: 7650 train videos, 850 validation videos

🎬 Processing TRAIN videos...
Train videos:   0%|                          | 4/7650 [00:05<2:23:27,  1.13s/it]

In [ ]:
# Cell 6: Cleanup & Check Disk Space
!rm -rf work_dirs/internvl2_5_4b_walk_lora/checkpoint-* 2>/dev/null || true
!rm -rf work_dirs/internvl2_5_4b_walk_lora/tmp-checkpoint-* 2>/dev/null || true
!rm -rf wandb/ 2>/dev/null || true

print("💾 Disk space:")
!df -h /workspace

In [6]:
!pip install "accelerate<1" \
    "bitsandbytes==0.42.0" \
    "decord" \
    "deepspeed>=0.13.5" \
    "einops==0.6.1" \
    "einops-exts==0.0.4" \
    "huggingface_hub" \
    "imageio" \
    "numpy==1.26.4" \
    "opencv-python-headless" \
    "orjson" \
    "peft==0.10.0" \
    "pycocoevalcap" \
    "pyyaml" \
    "scikit-learn>=1.2.2" \
    "scipy" \
    "sentencepiece==0.1.99" \
    "shortuuid" \
    "tensorboardX" \
    "termcolor" \
    "timm==0.9.12" \
    "tokenizers==0.15.1" \
    "transformers==4.37.2" \
    "yacs" \
    "wandb"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.0/105.0 MB 143.0 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.3/104.3 MB 160.7 MB/s eta 0:00:0000:0100:01
  Attempting uninstall: bitsandbytes
    Found existing installation: bitsandbytes 0.49.1
    Uninstalling bitsandbytes-0.49.1:
      Successfully uninstalled bitsandbytes-0.49.1

[notice] A new release of pip is available: 24.2 -> 26.0
[notice] To update, run: python -m pip install --upgrade pip


In [7]:
# Cell: Fix bitsandbytes for CUDA 12.x
!pip uninstall bitsandbytes -y
!pip install bitsandbytes>=0.43.0 --upgrade

# Verify installation
!python -c "import bitsandbytes; print('✅ bitsandbytes version:', bitsandbytes.__version__)"
!python -c "import torch; print('✅ CUDA available:', torch.cuda.is_available()); print('✅ CUDA version:', torch.version.cuda)"

Found existing installation: bitsandbytes 0.42.0
Uninstalling bitsandbytes-0.42.0:
  Successfully uninstalled bitsandbytes-0.42.0

[notice] A new release of pip is available: 24.2 -> 26.0
[notice] To update, run: python -m pip install --upgrade pip
✅ bitsandbytes version: 0.49.1
✅ CUDA available: True
✅ CUDA version: 12.4


In [8]:
# Cell 7: Run Training
!chmod +x train_walk_lora.sh
!./train_walk_lora.sh 2>&1 | tee training_log.txt

🚀 InternVL2.5-4B LoRA Training (FIXED)
✅ DeepSpeed config: ./zero_stage1_config.json
✅ Train meta: ./data/walk_vlm/walk_train_meta.json
✅ Eval meta: ./data/walk_vlm/walk_val_meta.json
⚙️  Key Settings:
   freeze_llm: False (LoRA will train)
   freeze_mlp: False
   freeze_backbone: True
   use_llm_lora: 16
petrel_client is not installed. If you read data locally instead of from ceph, ignore it.
petrel_client is not installed. Using PIL to load images.
df: /root/.triton/autotune: No such file or directory
02/03/2026 19:33:13 - WARNING - __main__ - Process rank: 0, device: cuda:0, n_gpu: 1distributed training: True, 16-bits training: False
02/03/2026 19:33:13 - INFO - __main__ - Training/evaluation parameters TrainingArguments(
_n_gpu=1,
adafactor=False,
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
bf16=True,
bf16_full_eval=False,
data_seed=None,
dataloader_drop_last=False,
dataloader_num_workers=4,
dataloader_persistent_workers=False,
dataloader_pin_m

In [13]:
# Cell 8: Upload Model to HuggingFace
!python upload_to_hf.py

Creating repo: blind-assist/internvl2-5-4b-walk-lora-v2-100...
🔄 Converting checkpoint to standard PEFT format...
   Loading: model-00001-of-00002.safetensors
   Loading: model-00002-of-00002.safetensors
   📊 Extracted 504 LoRA tensors
   📊 Target modules: {'o_proj', 'v_proj', 'down_proj', 'q_proj', 'gate_proj', 'up_proj', 'k_proj'}
   📊 LoRA rank: 16
   ✅ Saved PEFT adapter to work_dirs/internvl2_5_4b_walk_lora_peft
📝 Generating Model Card...
✅ Model card saved
📤 Uploading PEFT adapter from work_dirs/internvl2_5_4b_walk_lora_peft...
Processing Files (0 / 0)      : |                  |  0.00B /  0.00B            
New Data Upload               : |                  |  0.00B /  0.00B            

  ...adapter_model.safetensors:   4%|▌             | 2.53MB / 59.9MB            

Processing Files (0 / 1)      :   4%|▌             | 2.53MB / 59.9MB, 6.33MB/s  
New Data Upload               :   4%|▌             | 2.53MB / 59.9MB, 6.33MB/s  

Processing Files (0 / 1)      :  99%|█████████████▉|

In [18]:
# Cell 10: Verify Videos
%cd /workspace/InternVL/internvl_chat

# Check what's in my_videos folder
print("📹 Videos in my_videos folder:")
!ls -la my_videos/

# Count videos
print("\n📊 Total video files:")
!ls my_videos/*.mp4 my_videos/*.avi my_videos/*.mov 2>/dev/null | wc -l

/workspace/InternVL/internvl_chat
📹 Videos in my_videos folder:
total 42104
drwxrwxrwx  2 root root 2003633 Feb  2 17:35 .
drwxrwxrwx 12 root root 3002153 Feb  2 17:37 ..
-rw-rw-rw-  1 root root  759376 Feb  2 17:35 20240914_1abe4d6f616ccb2d8e15049136e1656d_4m28s.mp4
-rw-rw-rw-  1 root root 3677174 Feb  2 17:35 20240918-youtube_short_081e0a96bac802b988a1db9df310ddd1_1min03s.mp4
-rw-rw-rw-  1 root root 2372286 Feb  2 17:35 20240918-youtube_short_11bc0f3b2d7f68e62f5d61a10d2f8897_4min03s.mp4
-rw-rw-rw-  1 root root 1871479 Feb  2 17:35 20240918-youtube_short_1aec7c3e80c13cbb75f75f72333148bf_2min24s.mp4
-rw-rw-rw-  1 root root 2584601 Feb  2 17:35 20240918-youtube_short_1cb8f8832a143fde640ff92d7c656280_4m28s.mp4
-rw-rw-rw-  1 root root 2769889 Feb  2 17:35 20240918-youtube_short_2711f140bac74f83aa0a56272ec7f6de_3s.mp4
-rw-rw-rw-  1 root root 3417858 Feb  2 17:35 20240918-youtube_short_83bc88612ff6a00628af923c9ff461cb_2m9s.mp4
-rw-rw-rw-  1 root root 2418653 Feb  2 17:35 20240918-youtube_sh

In [10]:
# Cell 11: Verify Setup Before Testing
%cd /workspace/InternVL/internvl_chat

# Check if test script exists
print("📄 Test script:")
!ls -la test_finetuned_model.py 2>/dev/null || echo "❌ Script not found!"

# Check if videos exist
print("\n📹 Videos in my_videos folder:")
!ls -la my_videos/ 2>/dev/null || echo "❌ my_videos folder not found!"

# Count videos
print("\n📊 Total videos:")
!find my_videos/ -name "*.mp4" -o -name "*.avi" -o -name "*.mov" 2>/dev/null | wc -l

/workspace/InternVL/internvl_chat
📄 Test script:
-rw-rw-rw- 1 root root 32486 Feb  3 19:23 test_finetuned_model.py

📹 Videos in my_videos folder:
total 42102
drwxrwxrwx  2 root root 2003633 Feb  3 20:09 .
drwxrwxrwx 12 root root 3000709 Feb  3 20:09 ..
-rw-rw-rw-  1 root root  759376 Feb  3 20:09 20240914_1abe4d6f616ccb2d8e15049136e1656d_4m28s.mp4
-rw-rw-rw-  1 root root 3677174 Feb  3 20:09 20240918-youtube_short_081e0a96bac802b988a1db9df310ddd1_1min03s.mp4
-rw-rw-rw-  1 root root 2372286 Feb  3 20:09 20240918-youtube_short_11bc0f3b2d7f68e62f5d61a10d2f8897_4min03s.mp4
-rw-rw-rw-  1 root root 1871479 Feb  3 20:09 20240918-youtube_short_1aec7c3e80c13cbb75f75f72333148bf_2min24s.mp4
-rw-rw-rw-  1 root root 2584601 Feb  3 20:09 20240918-youtube_short_1cb8f8832a143fde640ff92d7c656280_4m28s.mp4
-rw-rw-rw-  1 root root 2769889 Feb  3 20:09 20240918-youtube_short_2711f140bac74f83aa0a56272ec7f6de_3s.mp4
-rw-rw-rw-  1 root root 3417858 Feb  3 20:09 20240918-youtube_short_83bc88612ff6a00628af923c

In [11]:
# Cell 12: Run Test
!python test_finetuned_model.py --input ./my_videos --output ./inference_results_finetuned

🔄 Loading base model: OpenGVLab/InternVL2_5-4B
/usr/local/lib/python3.11/dist-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
configuration_internvl_chat.py: 4.04kB [00:00, 7.64MB/s]
configuration_intern_vit.py: 5.55kB [00:00, 23.7MB/s]
A new version of the following files was downloaded from https://huggingface.co/OpenGVLab/InternVL2_5-4B:
- configuration_intern_vit.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
A new version of the following files was downloaded from https://huggingface.co/OpenGVLab/InternVL2_5-4B:
- configuration_internvl_chat.py
- configuration_intern_vit.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versi

In [29]:
# Cell: First check what's in the checkpoint folder
%cd /workspace/InternVL/internvl_chat

print("📁 Main output directory:")
!ls -la work_dirs/internvl2_5_4b_walk_lora/

print("\n📁 Checkpoint-50 directory:")
!ls -la work_dirs/internvl2_5_4b_walk_lora/checkpoint-50/

print("\n🔍 Looking for adapter_config.json:")
!find work_dirs/ -name "adapter_config.json" 2>/dev/null

/workspace/InternVL/internvl_chat
📁 Main output directory:
total 7323804
drwxrwxrwx 3 root root    3002125 Feb  2 17:06 .
drwxrwxrwx 3 root root    3002125 Feb  2 16:38 ..
-rw-rw-rw- 1 root root       1256 Feb  2 17:06 README.md
-rw-rw-rw- 1 root root        790 Feb  2 16:58 added_tokens.json
-rw-rw-rw- 1 root root        189 Feb  2 16:58 all_results.json
drwxrwxrwx 3 root root    3001428 Feb  2 16:56 checkpoint-50
-rw-rw-rw- 1 root root       5475 Feb  2 16:57 config.json
-rw-rw-rw- 1 root root        129 Feb  2 16:57 generation_config.json
-rw-rw-rw- 1 root root    1670344 Feb  2 16:58 merges.txt
-rw-rw-rw- 1 root root 4988318800 Feb  2 16:57 model-00001-of-00002.safetensors
-rw-rw-rw- 1 root root 2497013576 Feb  2 16:58 model-00002-of-00002.safetensors
-rw-rw-rw- 1 root root     147690 Feb  2 16:58 model.safetensors.index.json
-rw-rw-rw- 1 root root        744 Feb  2 16:58 special_tokens_map.json
-rw-rw-rw- 1 root root       7280 Feb  2 16:58 tokenizer_config.json
-rw-rw-rw- 1 root 

In [12]:
# Cell: Test with BASE model first (should work correctly)
!python test_finetuned_model.py --input ./my_videos --output ./inference_results_base --base

🔄 Loading base model: OpenGVLab/InternVL2_5-4B
/usr/local/lib/python3.11/dist-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Loading checkpoint shards: 100%|██████████████████| 2/2 [00:01<00:00,  1.34it/s]
/usr/local/lib/python3.11/dist-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
✅ Base model loaded!

📁 Input folder: ./my_videos
🚀 Processing 15 vide

In [27]:
# Cell 10: Check Local Trained Model
%cd /workspace/InternVL/internvl_chat

print("📁 Checking local model files:")
!ls -la work_dirs/internvl2_5_4b_walk_lora/

print("\n📄 Check for adapter_config.json (LoRA) or model files (merged):")
!cat work_dirs/internvl2_5_4b_walk_lora/adapter_config.json 2>/dev/null && echo "✅ This is a LoRA adapter" || echo "❌ No adapter_config.json"
!ls work_dirs/internvl2_5_4b_walk_lora/*.safetensors 2>/dev/null && echo "✅ Has safetensors files" || echo "❌ No safetensors"

print("\n📊 Training results:")
!cat work_dirs/internvl2_5_4b_walk_lora/train_results.json 2>/dev/null || echo "No train_results.json"

/workspace/InternVL/internvl_chat
📁 Checking local model files:
total 7323804
drwxrwxrwx 3 root root    3002125 Feb  2 17:06 .
drwxrwxrwx 3 root root    3002125 Feb  2 16:38 ..
-rw-rw-rw- 1 root root       1256 Feb  2 17:06 README.md
-rw-rw-rw- 1 root root        790 Feb  2 16:58 added_tokens.json
-rw-rw-rw- 1 root root        189 Feb  2 16:58 all_results.json
drwxrwxrwx 3 root root    3001428 Feb  2 16:56 checkpoint-50
-rw-rw-rw- 1 root root       5475 Feb  2 16:57 config.json
-rw-rw-rw- 1 root root        129 Feb  2 16:57 generation_config.json
-rw-rw-rw- 1 root root    1670344 Feb  2 16:58 merges.txt
-rw-rw-rw- 1 root root 4988318800 Feb  2 16:57 model-00001-of-00002.safetensors
-rw-rw-rw- 1 root root 2497013576 Feb  2 16:58 model-00002-of-00002.safetensors
-rw-rw-rw- 1 root root     147690 Feb  2 16:58 model.safetensors.index.json
-rw-rw-rw- 1 root root        744 Feb  2 16:58 special_tokens_map.json
-rw-rw-rw- 1 root root       7280 Feb  2 16:58 tokenizer_config.json
-rw-rw-rw- 1 

In [16]:
!python test_from_huggingface.py --input ./my_videos

🔄 Loading base model: OpenGVLab/InternVL2_5-4B
/usr/local/lib/python3.11/dist-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Loading checkpoint shards: 100%|██████████████████| 2/2 [00:01<00:00,  1.38it/s]
/usr/local/lib/python3.11/dist-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
🔄 Downloading LoRA adapter from: blind-assist/internvl2-5-4b-walk-lora